# 10c -- MTMC Stages 4-5: Association + Evaluation

**Prerequisite**: attach **10b's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-10b-stage-3-faiss-indexing" -> add`

**This is the iteration loop** -- edit the tuning params cell and re-run in ~6 min.
No GPU needed, no data download, no model inference.

| Stage | What | Time |
|---|---|---|
| 4 | Cross-camera association (AQE + Louvain graph clustering) | ~5 min |
| 5 | Evaluation: IDF1, MOTA, HOTA | ~1 min |

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _major, _minor = _cap.strip().split(".")
        _sm = int(_major) * 10 + int(_minor)
        if _sm < 70:
            print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) — installing compatible PyTorch ...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu124",
            ])
            print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

try:
    import trackeval; print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 Dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("trackeval", "trackeval"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Load Checkpoint from 10b

Finds `checkpoint.tar.gz` in `/kaggle/input/mtmc-10b-stage-3-faiss-indexing/` and extracts it.

In [ ]:
PREV_SLUG = "mtmc-10b-stage-3-faiss-indexing"
PREV_INPUT = Path("/kaggle/input") / PREV_SLUG

if not PREV_INPUT.exists():
    for p in Path("/kaggle/input").iterdir():
        if PREV_SLUG in p.name or "stage3" in p.name or "10b" in p.name:
            PREV_INPUT = p; break

cp = PREV_INPUT / "checkpoint.tar.gz"

# Fallback: download via Kaggle API if not found in mounted input
if not cp.exists():
    print(f"checkpoint.tar.gz not found at {cp} -- attempting API download")
    import subprocess as _sp
    _dl_dir = Path("/tmp/kaggle_dl")
    _dl_dir.mkdir(parents=True, exist_ok=True)
    _r = _sp.run(
        ["kaggle", "kernels", "output",
         "mrkdagods/mtmc-10b-stage-3-faiss-indexing",
         "--file", "checkpoint.tar.gz",
         "-p", str(_dl_dir)],
        capture_output=True, text=True
    )
    print(_r.stdout); print(_r.stderr)
    cp = _dl_dir / "checkpoint.tar.gz"

assert cp.exists(), f"checkpoint.tar.gz not found at {cp}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {cp.stat().st_size/1024**2:.1f} MB ...")
with tarfile.open(str(cp), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR  = EXTRACT_DIR / RUN_NAME
# GT annotations are in the repo under data/raw/cityflowv2/*/gt/gt.txt
# They were force-committed despite data/ being gitignored.
_repo_gt = PROJECT / "data" / "raw" / "cityflowv2"
if any((_repo_gt / cam / "gt" / "gt.txt").exists()
       for cam in ["S01_c001", "S01_c002", "S01_c003",
                   "S02_c006", "S02_c007", "S02_c008"]):
    GT_DIR = str(_repo_gt)
    print(f"\u2713 GT annotations at {GT_DIR}")
else:
    GT_DIR = ""
    print("WARNING: GT files not found in repo — eval will skip metrics")
print(f"\u2713 Checkpoint extracted -- run: {RUN_NAME}")
for s in ["stage1", "stage2", "stage3"]:
    d = RUN_DIR / s
    if d.exists(): print(f"  {s}: {len(list(d.rglob('*')))} files")
print(f"  GT dir: {GT_DIR}")

## 3. Tuning Parameters

**Edit these values** then re-run the cells below (~6 min). No need to re-run 10a or 10b.

In [ ]:
# ---- Stage 4: Cross-camera association -------------------------------------
# v46 optimized config (local best: IDF1=0.8297)
AQE_K             = 3     # v46: 2 (optimal after QE self-exclusion fix)
SIM_THRESH        = 0.53  # v46: 0.53 (optimal from sweep)
ALGORITHM         = "conflict_free_cc"   # v67: +0.21pp over connected_components
LOUVAIN_RES       = 0.70  # fallback for community_detection

# Weights — v46 defaults (appearance-heavy for vehicles)
APPEARANCE_WEIGHT = 0.75  # v46: CityFlowV2 vehicle config
HSV_WEIGHT        = 0.0   # v46: disabled (hurts vehicles)
ST_WEIGHT         = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)

# Bridge pruning: v46 found pruning HURTS (-1.4pp) → disabled
BRIDGE_PRUNE      = 0.0   # v46: 0.0 (pruning hurts by -1.4pp)
MAX_COMP_SIZE     = 12

# Intra-camera merge: +1.0pp improvement
INTRA_MERGE       = True
INTRA_MERGE_THRESH = 0.75
INTRA_MERGE_GAP   = 60    # seconds

# v52: Score-level fusion with OSNet secondary embeddings
FUSION_WEIGHT     = 0.10   # v53: 10% secondary (was 0.3 in v52)

# v54: SOTA techniques (AIC21 1st place)
# Camera bias: learn per-camera-pair similarity offsets from clusters
CAMERA_BIAS       = False
CAMERA_BIAS_ITERS = 2     # iterative: cluster → learn bias → re-cluster
# Zone model: penalize impossible transitions, boost valid ones
ZONE_MODEL        = False
ZONE_BONUS        = 0.06  # boost valid zone transitions
ZONE_PENALTY      = 0.04  # penalize invalid transitions
# Hierarchical: multi-pass centroid expansion to recover orphans
HIERARCHICAL      = False
HIER_CENTROID_TH  = 0.35  # orphan → cluster threshold
HIER_MERGE_TH     = 0.35  # cluster ↔ cluster merge threshold
HIER_ORPHAN_TH    = 0.30  # orphan ↔ orphan final threshold

# ---- Stage 5: Evaluation ----------------------------------------------------
# v46: MTMC_ONLY=False keeps single-cam trajectories (+5pp IDF1).
# Single-cam tracks ARE real vehicles; dropping them loses 29/240=12% GT.
MTMC_ONLY = False

print("Stage 4 params (v46 optimized):")
print(f"  aqe_k={AQE_K}  sim_thresh={SIM_THRESH}  algorithm={ALGORITHM}  appearance_weight={APPEARANCE_WEIGHT}")
print(f"  hsv_weight={HSV_WEIGHT}  st_weight={ST_WEIGHT}  bridge_prune={BRIDGE_PRUNE}  max_comp_size={MAX_COMP_SIZE}")
print(f"  intra_merge={INTRA_MERGE}  intra_thresh={INTRA_MERGE_THRESH}  intra_gap={INTRA_MERGE_GAP}")
print(f"  fusion_weight={FUSION_WEIGHT}")
print(f"  camera_bias={CAMERA_BIAS}  zone_model={ZONE_MODEL}  hierarchical={HIERARCHICAL}")
print(f"  zone_bonus={ZONE_BONUS}  zone_penalty={ZONE_PENALTY}")
print(f"  hier: centroid={HIER_CENTROID_TH}  merge={HIER_MERGE_TH}  orphan={HIER_ORPHAN_TH}")
print(f"Stage 5: mtmc_only_submission={MTMC_ONLY}, stationary_filter=True")

## 4. Run Stages 4-5

In [ ]:
os.chdir(str(PROJECT))
cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "4,5",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", f"stage4.association.query_expansion.k={AQE_K}",
    "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
    "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
    "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
    "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
    "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
    "--override", f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
    "--override", "stage4.association.mutual_nn.top_k_per_query=20",
    "--override", "stage4.association.fic.enabled=true",
    "--override", "stage4.association.fic.regularisation=3.0",
    "--override", "stage4.association.fac.enabled=false",
    "--override", "stage4.association.fac.knn=20",
    "--override", "stage4.association.fac.learning_rate=0.5",
    "--override", "stage4.association.fac.beta=0.08",
    # v52: score-level fusion with OSNet secondary embeddings
    "--override", f"stage4.association.secondary_embeddings.path={RUN_DIR}/stage2/embeddings_secondary.npy",
    "--override", f"stage4.association.secondary_embeddings.weight={FUSION_WEIGHT}",
    # v54: SOTA — camera distance bias
    "--override", f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
    "--override", f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
    # v54: SOTA — zone-based transition scoring
    "--override", f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
    "--override", "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
    "--override", f"stage4.association.zone_model.bonus={ZONE_BONUS}",
    "--override", f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
    # v54: SOTA — hierarchical centroid expansion
    "--override", f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
    "--override", f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
    "--override", f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
    "--override", f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
    "--override", "stage4.association.hierarchical.max_merge_size=12",
    "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
    "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
    "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
    "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
    "--override", "stage5.stationary_filter.enabled=true",
    "--override", "stage5.stationary_filter.min_displacement_px=150",
    "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
    "--override", "stage5.min_submission_confidence=0.15",
    "--override", "stage5.cross_id_nms_iou=0.40",
    "--override", "stage5.min_trajectory_confidence=0.30",
    "--override", "stage5.min_trajectory_frames=25",
    "--override", "stage5.track_edge_trim.enabled=false",
    "--override", "stage5.track_smoothing.enabled=false",
    "--override", "stage5.gt_frame_clip=true",
    "--override", "stage5.gt_zone_filter=true",
]
if GT_DIR:
    cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
else:
    print("WARNING: GT_DIR is empty — eval will skip metric computation")
print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"\u2717 FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"\u2713 Stages 4-5 done in {elapsed/60:.1f} min")

## 5. Results

In [ ]:
stage5_dir = RUN_DIR / "stage5"

def _pct(v):
    return f"{v:.1%}" if isinstance(v, float) else str(v)

metrics_files = sorted(stage5_dir.glob("metrics_*.json")) if stage5_dir.exists() else []
if metrics_files:
    print("=" * 65)
    print("EVALUATION RESULTS")
    print("=" * 65)
    for mf in metrics_files:
        m = json.loads(mf.read_text())
        m = m.get("metrics", m)
        cam = mf.stem.replace("metrics_", "")
        idf1 = _pct(m.get("IDF1", m.get("idf1", "?")))
        mota = _pct(m.get("MOTA", m.get("mota", "?")))
        hota = _pct(m.get("HOTA", m.get("hota", "?")))
        idsw = m.get("ID_Sw", m.get("id_switches", "?"))
        print(f"  {cam:12s}  IDF1={idf1}  MOTA={mota}  HOTA={hota}  IDsw={idsw}")

for fname in ["summary.json", "evaluation_report.json"]:
    sf = stage5_dir / fname
    if sf.exists():
        s = json.loads(sf.read_text())
        print("-" * 65 + "\n  GLOBAL:")
        for k in ["IDF1","MOTA","HOTA","ID_Sw","idf1","mota","hota","id_switches","mtmc_idf1"]:
            v = s.get(k)
            if v is not None: print(f"    {k}: {_pct(v)}")
        break

if not metrics_files:
    print("No metrics files found -- check stage5 output dir:", stage5_dir)

# Copy evaluation report to /kaggle/working/ for easy download
import shutil as _shutil
for _fname in ["evaluation_report.json", "summary.json"]:
    _src = stage5_dir / _fname
    if _src.exists():
        _shutil.copy2(str(_src), str(Path("/kaggle/working") / _fname))
        print(f"Copied {_fname} to /kaggle/working/")


## 6. Automated Parameter Scan (optional)

Runs stages 4-5 across a grid of parameter values and reports the best combination.
Each combination takes ~2 min → a 12-point scan takes ~24 min total.

**Comment out this cell if you just want the single run above.**

In [ ]:
# ============================================================
# v70: FIC REGULARIZATION + SIM_THRESH + EXTENDED FRAMES SWEEP
# One-at-a-time sweeps to map parameter landscape
# Baseline: conflict_free_cc, frames=25, nms=0.40, fic_reg=3.0, sim=0.53
# ============================================================
SCAN_ENABLED = True

if SCAN_ENABLED:
    experiments = []

    # Defaults (v69 best)
    defaults = {
        "fic_reg": 3.0,
        "sim_thresh_v": SIM_THRESH,  # 0.53
        "min_traj_frames_v": 25,
    }

    # Baseline experiment
    experiments.append({"name": "baseline", **defaults})

    # FIC regularization sweep (hold sim=0.53, frames=25)
    for fic in [0.1, 0.5, 1.0, 2.0, 5.0, 8.0, 15.0]:
        if fic == defaults["fic_reg"]:
            continue
        experiments.append({"name": f"fic_{fic}".replace(".", "p"),
            "fic_reg": fic, "sim_thresh_v": defaults["sim_thresh_v"],
            "min_traj_frames_v": defaults["min_traj_frames_v"]})

    # sim_thresh sweep (hold fic=3.0, frames=25)
    for sim in [0.45, 0.48, 0.50, 0.52, 0.54, 0.55, 0.58, 0.60]:
        experiments.append({"name": f"sim_{sim}".replace(".", "p"),
            "fic_reg": defaults["fic_reg"], "sim_thresh_v": sim,
            "min_traj_frames_v": defaults["min_traj_frames_v"]})

    # Extended frames (hold fic=3.0, sim=0.53)
    for f in [30, 35, 40]:
        experiments.append({"name": f"frames_{f}",
            "fic_reg": defaults["fic_reg"], "sim_thresh_v": defaults["sim_thresh_v"],
            "min_traj_frames_v": f})

    print(f"Running {len(experiments)} experiments ...")

    results = []
    for exp in experiments:
        scan_run = f"scan_{exp['name']}"

        # Symlink upstream stages
        scan_dir = DATA_OUT / scan_run
        scan_dir.mkdir(parents=True, exist_ok=True)
        for stage_sub in ("stage1", "stage2", "stage3"):
            src = DATA_OUT / RUN_NAME / stage_sub
            dst = scan_dir / stage_sub
            if src.exists() and not dst.exists():
                dst.symlink_to(src)

        cmd_scan = [
            sys.executable, "scripts/run_pipeline.py",
            "--config", "configs/default.yaml",
            "--dataset-config", "configs/datasets/cityflowv2.yaml",
            "--stages", "4,5",
            "--override", f"project.run_name={scan_run}",
            "--override", f"project.output_dir={DATA_OUT}",
            # Stage 4 params (v69 best + sweep variables)
            "--override", f"stage4.association.query_expansion.k={AQE_K}",
            "--override", f"stage4.association.graph.similarity_threshold={exp['sim_thresh_v']}",
            "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
            "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
            "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
            "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
            "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
            "--override", "stage4.association.mutual_nn.top_k_per_query=20",
            "--override", "stage4.association.fic.enabled=true",
            "--override", f"stage4.association.fic.regularisation={exp['fic_reg']}",
            "--override", "stage4.association.fac.enabled=false",
            "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
            "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
            "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
            "--override", f"stage4.association.secondary_embeddings.path={RUN_DIR}/stage2/embeddings_secondary.npy",
            "--override", f"stage4.association.secondary_embeddings.weight={FUSION_WEIGHT}",
            "--override", f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
            "--override", f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
            "--override", f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
            # Stage 5: v69 best + sweep variable
            "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
            "--override", "stage5.stationary_filter.enabled=true",
            "--override", "stage5.stationary_filter.min_displacement_px=150",
            "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
            "--override", "stage5.min_submission_confidence=0.15",
            "--override", "stage5.cross_id_nms_iou=0.40",
            "--override", "stage5.min_trajectory_confidence=0.30",
            "--override", f"stage5.min_trajectory_frames={exp['min_traj_frames_v']}",
            "--override", "stage5.track_edge_trim.enabled=false",
            "--override", "stage5.track_smoothing.enabled=false",
            "--override", "stage5.gt_frame_clip=true",
            "--override", "stage5.gt_zone_filter=true",
        ]
        if GT_DIR:
            cmd_scan += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
        t0 = time.time()
        r = subprocess.run(cmd_scan, cwd=str(PROJECT), capture_output=True)
        elapsed = time.time() - t0

        # Read metrics
        report = DATA_OUT / scan_run / "stage5" / "evaluation_report.json"
        idf1 = mota = hota = mtmc_idf1 = 0.0
        id_switches = 0
        if report.exists():
            rp = json.loads(report.read_text())
            m = rp.get("metrics", rp)
            idf1 = m.get("IDF1", m.get("idf1", 0.0))
            mota = m.get("MOTA", m.get("mota", 0.0))
            hota = m.get("HOTA", m.get("hota", 0.0))
            mtmc_idf1 = m.get("mtmc_idf1", 0.0)
            id_switches = m.get("id_switches", 0)

        results.append({
            "name": exp["name"],
            "fic_reg": exp["fic_reg"], "sim_thresh": exp["sim_thresh_v"],
            "frames": exp["min_traj_frames_v"],
            "IDF1": idf1, "MOTA": mota, "HOTA": hota,
            "mtmc_idf1": mtmc_idf1, "id_switches": id_switches,
            "time": elapsed
        })
        status = "OK" if r.returncode == 0 else "FAIL"
        print(f"  [{status}] {exp['name']:20s} fic={exp['fic_reg']:5.1f} sim={exp['sim_thresh_v']:.2f} "
              f"frames={exp['min_traj_frames_v']:2d} -> IDF1={idf1:.4f} mtmc={mtmc_idf1:.4f} "
              f"HOTA={hota:.4f} ids={id_switches} ({elapsed:.0f}s)")

    # Sort by mtmc_idf1
    results.sort(key=lambda x: x["mtmc_idf1"], reverse=True)
    print("\n" + "=" * 100)
    print("v70 SWEEP RESULTS (sorted by mtmc_idf1)")
    print("=" * 100)
    print(f"{'Name':<20s} {'FIC':>5s} {'Sim':>5s} {'Frm':>4s} {'IDF1':>7s} {'mtmc_idf1':>10s} {'HOTA':>7s} {'IDs':>5s}")
    for r2 in results:
        marker = " <--" if r2["name"] == "baseline" else ""
        print(f"{r2['name']:<20s} {r2['fic_reg']:>5.1f} {r2['sim_thresh']:>5.2f} {r2['frames']:>4d} "
              f"{r2['IDF1']:>7.4f} {r2['mtmc_idf1']:>10.4f} {r2['HOTA']:>7.4f} {r2['id_switches']:>5d}{marker}")
    best = results[0]
    print(f"\nBEST: {best['name']} (fic={best['fic_reg']} sim={best['sim_thresh']} frames={best['frames']}) "
          f"-> mtmc_idf1={best['mtmc_idf1']:.4f} IDF1={best['IDF1']:.4f}")

    # Per-param sensitivity
    for param_name, default_val in [("fic_reg", 3.0), ("sim_thresh", 0.53), ("frames", 25)]:
        print(f"\n--- {param_name} sensitivity (default={default_val}) ---")
        for r2 in sorted(
            [r2 for r2 in results
             if all(r2[p] == dv for p, dv in [("fic_reg", 3.0), ("sim_thresh", 0.53), ("frames", 25)] if p != param_name)],
            key=lambda x: x["mtmc_idf1"], reverse=True
        ):
            marker = " <-- default" if r2[param_name] == default_val else ""
            print(f"  {param_name}={r2[param_name]:<8} mtmc_idf1={r2['mtmc_idf1']:.4f} IDF1={r2['IDF1']:.4f} "
                  f"HOTA={r2['HOTA']:.4f} ids={r2['id_switches']}{marker}")

    # Save
    import json as _json
    results_path = DATA_OUT / "scan_results.json"
    with open(results_path, "w") as _f:
        _json.dump({"scan_type": "fic_sim_frames_v70", "results": results}, _f, indent=2)
    print(f"\nSaved to {results_path}")
    import shutil as _shutil
    _out = Path("/kaggle/working")
    _shutil.copy2(str(results_path), str(_out / "scan_results.json"))
else:
    print("Scan disabled. Set SCAN_ENABLED = True to run v70 sweep.")

## 7. Feature A/B Tests

Tests untried association features vs the v46 baseline.
- **CSLS**: Cross-domain Similarity Local Scaling — hubness reduction to penalize 'universal hub' embeddings

In [ ]:
# --- Feature A/B tests ---
# Tests CSLS hubness reduction (last untested stage4 feature)
FEATURE_TEST_ENABLED = False

if FEATURE_TEST_ENABLED:
    # Define experiments: (tag, extra_overrides_dict)
    feature_experiments = [
        # --- temporal_split: split clusters at time gaps (targets conflation) ---
        ("tsplit_gap60_t050", {"stage4.association.temporal_split.enabled": "true",
                               "stage4.association.temporal_split.min_gap": "60",
                               "stage4.association.temporal_split.split_threshold": "0.50"}),
        ("tsplit_gap45_t045", {"stage4.association.temporal_split.enabled": "true",
                               "stage4.association.temporal_split.min_gap": "45",
                               "stage4.association.temporal_split.split_threshold": "0.45"}),
        ("tsplit_gap30_t040", {"stage4.association.temporal_split.enabled": "true",
                               "stage4.association.temporal_split.min_gap": "30",
                               "stage4.association.temporal_split.split_threshold": "0.40"}),
        # --- cluster_verify: eject weakly-connected cluster members ---
        ("cverify_030", {"stage4.association.cluster_verify.enabled": "true",
                         "stage4.association.cluster_verify.min_connectivity": "0.30"}),
        ("cverify_035", {"stage4.association.cluster_verify.enabled": "true",
                         "stage4.association.cluster_verify.min_connectivity": "0.35"}),
        ("cverify_025", {"stage4.association.cluster_verify.enabled": "true",
                         "stage4.association.cluster_verify.min_connectivity": "0.25"}),
        # --- combined: temporal_split + cluster_verify ---
        ("tsplit60_cverify030", {"stage4.association.temporal_split.enabled": "true",
                                 "stage4.association.temporal_split.min_gap": "60",
                                 "stage4.association.temporal_split.split_threshold": "0.50",
                                 "stage4.association.cluster_verify.enabled": "true",
                                 "stage4.association.cluster_verify.min_connectivity": "0.30"}),
    ]
    feat_results = []

    for tag, extra_overrides in feature_experiments:
        feat_run = f"{RUN_NAME}_{tag}"
        feat_dir = DATA_OUT / feat_run
        feat_dir.mkdir(parents=True, exist_ok=True)
        for stage_sub in ("stage1", "stage2", "stage3"):
            src = RUN_DIR / stage_sub
            dst = feat_dir / stage_sub
            if src.exists() and not dst.exists():
                dst.symlink_to(src)

        st_w = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)
        cmd_feat = [
            sys.executable, "scripts/run_pipeline.py",
            "--config", "configs/default.yaml",
            "--dataset-config", "configs/datasets/cityflowv2.yaml",
            "--stages", "4,5",
            "--override", f"project.run_name={feat_run}",
            "--override", f"project.output_dir={DATA_OUT}",
            "--override", f"stage4.association.query_expansion.k={AQE_K}",
            "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
            "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
            "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
            "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
            "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.spatiotemporal={st_w}",
            "--override", "stage4.association.weights.length_weight_power=0.5",
            "--override", "stage4.association.mutual_nn.top_k_per_query=20",
            "--override", "stage4.association.fic.enabled=true",
            "--override", "stage4.association.fic.regularisation=3.0",
            "--override", "stage4.association.fac.enabled=false",
            "--override", "stage4.association.fac.knn=20",
            "--override", "stage4.association.fac.learning_rate=0.5",
            "--override", "stage4.association.fac.beta=0.08",
            "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
            "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
            "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
            "--override", "stage4.association.gallery_expansion.enabled=true",
            "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
            "--override", "stage5.stationary_filter.enabled=true",
            "--override", "stage5.stationary_filter.min_displacement_px=150",
            "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
            "--override", "stage5.min_submission_confidence=0.15",
            "--override", "stage5.cross_id_nms_iou=0.35",
            "--override", "stage5.min_trajectory_confidence=0.30",
            "--override", "stage5.min_trajectory_frames=10",
            "--override", "stage5.track_edge_trim.enabled=false",
            "--override", "stage5.track_smoothing.enabled=false",
            "--override", "stage5.gt_frame_clip=true",
            "--override", "stage5.gt_zone_filter=true",
        ]
        # Add feature-specific overrides
        for k, v in extra_overrides.items():
            cmd_feat += ["--override", f"{k}={v}"]
        if GT_DIR:
            cmd_feat += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]

        t0 = time.time()
        r = subprocess.run(cmd_feat, cwd=str(PROJECT), capture_output=True)
        elapsed = time.time() - t0

        report = DATA_OUT / feat_run / "stage5" / "evaluation_report.json"
        idf1 = mota = hota = mtmc_idf1 = 0.0
        n_frag = n_conf = 0
        if report.exists():
            rp = json.loads(report.read_text())
            m = rp.get("metrics", rp)
            idf1 = m.get("IDF1", m.get("idf1", 0.0))
            mota = m.get("MOTA", m.get("mota", 0.0))
            hota = m.get("HOTA", m.get("hota", 0.0))
            mtmc_idf1 = m.get("mtmc_idf1", idf1)
            details = rp.get("details", {})
            n_frag = details.get("n_fragmented_gt", 0)
            n_conf = details.get("n_conflated_pred", 0)
        status = "OK" if r.returncode == 0 else "FAIL"
        feat_results.append({"tag": tag, "IDF1": idf1, "mtmc_idf1": mtmc_idf1,
                             "MOTA": mota, "HOTA": hota, "frag": n_frag, "conf": n_conf, "time": elapsed})
        print(f"  [{status}] {tag}: IDF1={idf1:.3f} mtmc_idf1={mtmc_idf1:.3f} MOTA={mota:.3f} HOTA={hota:.3f} frag={n_frag} conf={n_conf} ({elapsed/60:.1f} min)")

    print("\n" + "=" * 80)
    print("FEATURE A/B TEST RESULTS")
    print("=" * 80)
    print(f"  {'tag':<25} {'IDF1':>7} {'mtmc':>7} {'MOTA':>7} {'HOTA':>7} {'frag':>5} {'conf':>5}")
    print(f"  {'baseline (v46)' :<25} {'---':>7} {'---':>7} {'---':>7} {'---':>7} {'---':>5} {'---':>5}  (see section 5)")
    for r2 in sorted(feat_results, key=lambda x: x["IDF1"], reverse=True):
        print(f"  {r2['tag']:<25} {r2['IDF1']:>7.3f} {r2['mtmc_idf1']:>7.3f} {r2['MOTA']:>7.3f} {r2['HOTA']:>7.3f} {r2['frag']:>5} {r2['conf']:>5}")
else:
    print("Feature test disabled. Set FEATURE_TEST_ENABLED = False to run.")